# 02 · Limpieza

**Objetivo:** aplicar las decisiones de limpieza y guardar el dataset limpio
**Preguntas que responde:** base para P1–P6
**Entrada:** `data/raw/*.csv`
**Salida:** `data/processed/precios_limpio.parquet`
**Decisiones tomadas:** D-09 a D-19 en `docs/decisiones.md`

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.limpieza import cargar_tipado, agregar_derivadas, limpiar, finalizar, validar

df = agregar_derivadas(cargar_tipado("../data/raw"))
df, registro = limpiar(df)
df = finalizar(df)
validar(df)

print(registro.to_string(index=False))
print(df.shape, f"{df.memory_usage(deep=True).sum() / 1024**2:.0f} MB")
print(df.dtypes)

                           paso   filas
                filas iniciales 1297535
filas sin transacción excluidas     201
                  filas finales 1297334
(1297334, 21) 97 MB
fecha           datetime64[us]
id_region             category
region                category
mercado               category
subsector             category
producto              category
variedad              category
calidad               category
unidad                category
origen                category
volumen                  int64
precio_min             float64
precio_max             float64
precio_prom            float64
anio                     int32
mes                      int32
origen_tipo           category
tipo_unidad           category
kilos_unidad           float64
precio_kg              float64
es_rm                     bool
dtype: object


In [2]:
print(df["tipo_unidad"].value_counts(normalize=True).round(3))
print("Filas con precio_kg:", round(df["precio_kg"].notna().mean(), 3))
print(df["origen_tipo"].value_counts())
print("Proporción de filas en la RM:", round(df["es_rm"].mean(), 3))

tipo_unidad
kilos_envase             0.631
unidades                 0.284
kilo                     0.048
kilo_vol_unidades        0.020
rango                    0.009
unidades_vol_unidades    0.008
otro                     0.000
Name: proportion, dtype: float64
Filas con precio_kg: 0.699
origen_tipo
Nacional      1165111
Extranjero     132223
Name: count, dtype: int64
Proporción de filas en la RM: 0.389


In [1]:
from pathlib import Path
Path("../data/processed").mkdir(parents=True, exist_ok=True)

In [3]:
df.to_parquet("../data/processed/precios_limpio.parquet", index=False)

chk = pd.read_parquet("../data/processed/precios_limpio.parquet")
print(chk.shape, "tipos conservados:", chk.dtypes.equals(df.dtypes))

(1297334, 21) tipos conservados: True
